In [5]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoConfig
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification


num_labels = 11

checkpoint = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels
)


training_set = pd.read_csv("/content/test.csv")
test_set = pd.read_csv("/content/test.csv")
validation_set = pd.read_csv("/content/validation.csv")

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
#tokenizaiton
from datasets import Dataset

training_set_hf = Dataset.from_pandas(training_set)
validation_set_hf = Dataset.from_pandas(validation_set)
test_set_hf = Dataset.from_pandas(test_set)

def tokenize_function(training):
    return tokenizer(training["text"], truncation=True, max_length=256)


training_tokens = training_set_hf.map(tokenize_function, batched = True)
test_tokens = test_set_hf.map(tokenize_function, batched=True)
validation_tokens = validation_set_hf.map(tokenize_function, batched=True)

Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

Map:   0%|          | 0/9613 [00:00<?, ? examples/s]

In [8]:
cols_to_keep = {"input_ids", "attention_mask", "labels"}

remove_cols = [
    col for col in training_tokens.column_names
    if col not in cols_to_keep
]

training_tokens = training_tokens.remove_columns(remove_cols)
validation_tokens = validation_tokens.remove_columns(remove_cols)
test_tokens = test_tokens.remove_columns(remove_cols)

In [9]:
from huggingface_hub import notebook_login

notebook_login()

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

In [11]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)



training_args = TrainingArguments(
    output_dir="notebooks/retrain_results",  # output directory
    num_train_epochs=3,  # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=64,  # batch size for evaluation
    warmup_steps=500,  # number of warmup steps for learning rate scheduler
    weight_decay=0.01,  # strength of weight decay
    logging_dir="notebooks/retrain_logs",  # directory for storing logs
    logging_steps=10,
    push_to_hub=True,
    hub_model_id="Joshua2565/complaint-radar-distilbert"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_tokens,  # training dataset
    eval_dataset=validation_tokens,  # evaluation dataset
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
trainer.train()

Step,Training Loss
10,2.376527
20,2.359784
30,2.345144
40,2.327775
50,2.306026
60,2.263680
70,2.230199
80,2.173930
90,2.154818
100,2.093805


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1968, training_loss=0.7497299816913721, metrics={'train_runtime': 803.005, 'train_samples_per_second': 39.187, 'train_steps_per_second': 2.451, 'total_flos': 2084357141593440.0, 'train_loss': 0.7497299816913721, 'epoch': 3.0})

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [13]:
save_dir = "complaint-radar-distilbert"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...results/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...results/model.safetensors:   0%|          |  575kB /  268MB            

('complaint-radar-distilbert/tokenizer_config.json',
 'complaint-radar-distilbert/tokenizer.json')

In [14]:
model.push_to_hub("complaint-radar-distilbert")
tokenizer.push_to_hub("complaint-radar-distilbert")

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r26trdn/model.safetensors:  36%|###5      | 96.0MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Joshua2565/complaint-radar-distilbert/commit/9b54c76ab1ab73a214580f0575a7edf863b00b12', commit_message='Upload tokenizer', commit_description='', oid='9b54c76ab1ab73a214580f0575a7edf863b00b12', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Joshua2565/complaint-radar-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='Joshua2565/complaint-radar-distilbert'), pr_revision=None, pr_num=None)

In [15]:
trainer.push_to_hub("complaint-radar-distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...results/model.safetensors:  36%|###5      | 96.0MB /  268MB            

  ...results/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Joshua2565/complaint-radar-distilbert/commit/9b54c76ab1ab73a214580f0575a7edf863b00b12', commit_message='complaint-radar-distilbert', commit_description='', oid='9b54c76ab1ab73a214580f0575a7edf863b00b12', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Joshua2565/complaint-radar-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='Joshua2565/complaint-radar-distilbert'), pr_revision=None, pr_num=None)

In [16]:
print(trainer.compute_metrics)

<function compute_metrics at 0x7934c62fc040>


In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo_id = "Joshua2565/complaint-radar-distilbert"


tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)

config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [18]:
import pandas as pd
import numpy as np
from datasets import Dataset

test_set = pd.read_csv("/content/test.csv")

test_set_hf = Dataset.from_pandas(test_set)

In [19]:
def tokenize_func(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)


test_tokens = test_set_hf.map(tokenize_func, batched=True)

Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

In [22]:
keep_cols = {"input_ids", "attention_mask", "labels"}

test_tokens = test_tokens.remove_columns(
    [col for col in test_tokens.column_names if col not in keep_cols]
)


In [23]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="eval_results",
    per_device_eval_batch_size=64,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [24]:
pred_output = trainer.predict(test_tokens)

In [25]:
import numpy as np

logits = pred_output.predictions
y_true = pred_output.label_ids

y_pred = np.argmax(logits, axis=-1)

In [26]:
print(y_true)

[2 3 5 ... 3 3 9]


In [27]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.9216321860997235
Macro F1: 0.7343408005720228
Weighted F1: 0.9169612067697215
